# CSCI 447/547 Hackathon 6: Support Vector Machines and Decision Trees

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the workbook together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

##### In this exercise, you will train support vector machines and decision trees, explore how their hyperparameters shape their decision boundaries, and compare them to logistic regression.

---

### Google Colab Instructions

If you are using Google Colab <a href="https://colab.research.google.com/github/lucywowen/csci547_ML/blob/main/examples/hackathon6-SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Google Colab"/></a> you do not need to download anything: every dataset in this notebook is built into `scikit-learn`.

---

### Local (VSCode/Jupyter Lab) Version

To get the latest version of this notebook **without overwriting your work on other hackathons**, open a terminal in your cloned `csci547_ML` folder and run:

```bash
git fetch
git checkout origin/main -- examples/hackathon6-SVM.ipynb
```

`git fetch` downloads the latest version of the repository without changing any of your files, then `git checkout` replaces only this notebook. *(This overwrites any changes you have made to this notebook, so if you have already started it, make a copy first!)*

Make sure that your `uv` virtual environment is enabled.

---

## 1. Support Vector Machines

Logistic regression and neural networks learn their weights by minimizing a loss averaged over *every* training point. A **Support Vector Machine (SVM)** takes a geometric approach instead: find the boundary that separates the classes with the widest possible **margin**, the empty "street" between the two classes.

**The Linear SVM:** Like logistic regression, a linear SVM computes $f(\mathbf{x}) = \mathbf{w}^T\mathbf{x} + b$ and predicts one class when $f(\mathbf{x}) \geq 0$ and the other when $f(\mathbf{x}) < 0$. The edges of the margin are where $f(\mathbf{x}) = \pm 1$, so the width of the margin is $\dfrac{2}{\Vert\mathbf{w}\Vert}$: smaller weights mean a wider margin.

**The Soft-Margin Objective:** Real data is rarely perfectly separable, so we allow some points inside the margin (or even on the wrong side) but charge a penalty for them. With labels $y \in \{-1, +1\}$, the SVM minimizes:

$$\min_{\mathbf{w}, b}\; \underbrace{\frac{1}{2}\Vert\mathbf{w}\Vert^2}_{\text{wide margin}} \;+\; C\sum_{i=1}^m \underbrace{\max\big(0,\; 1 - y^{(i)}(\mathbf{w}^T\mathbf{x}^{(i)} + b)\big)}_{\text{hinge loss}}$$

* The **hinge loss** is 0 for a point that is correctly classified *and* outside the margin, and grows linearly the further a point sits inside the margin or on the wrong side.
* The hyperparameter **C** controls the trade-off between a wide margin and margin violations.

**Support Vectors:** The training points that sit on or inside the margin are called **support vectors**. They are the *only* points that determine the boundary: you could move or delete any other training point (without crossing the margin) and the SVM would not change at all.

### 1.1 Load the Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style='white')

from sklearn import datasets
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

### 1.2 A Helper for Plotting Decision Boundaries

We will draw a lot of decision boundaries in this notebook, so here is a helper function (handled for you). It colors each region of the plot by the class the model predicts there, then draws the data points on top. With `show_margin=True`, it also draws an SVM's margin as dashed lines and circles its support vectors in red.

In [ ]:
def plot_decision_boundary(model, X, y, ax, title='', show_margin=False):
    """
    Plots a fitted model's decision regions for 2D data.
    model: a fitted classifier
    X: the data points, shape (m, 2)
    y: the class labels, shape (m,)
    ax: the matplotlib axis to draw on
    show_margin: if True, also draw an SVM's margin and circle its support vectors
    """
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # color each point of the grid by the class the model predicts there
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=plt.cm.viridis, alpha=0.3)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap=plt.cm.viridis, edgecolors='k', s=30)

    if show_margin:
        # the boundary is where f(x) = 0, and the edges of the margin are where f(x) = -1 and +1
        f = model.decision_function(grid).reshape(xx.shape)
        ax.contour(xx, yy, f, levels=[-1, 0, 1], colors='k', linestyles=['--', '-', '--'])
        sv = model.support_vectors_
        ax.scatter(sv[:, 0], sv[:, 1], s=120, facecolors='none', edgecolors='r', linewidths=1.5)

    ax.set(xticks=[], yticks=[], title=title)

### 1.3 Look at the Data

We'll start with two blobs of points that are *almost* (but not quite) separable by a straight line.

In [ ]:
blobs_X, blobs_y = datasets.make_blobs(n_samples=100, centers=2, cluster_std=1.4, random_state=3)

plt.figure(figsize=(7, 5))
plt.scatter(blobs_X[:, 0], blobs_X[:, 1], c=blobs_y, cmap=plt.cm.viridis, edgecolors='k')
plt.title('Two Blobs')
plt.show()

### Think-Pair-Share 1: The C Hyperparameter
**Context:** In the soft-margin objective, $C$ multiplies the hinge loss (the penalty for points inside the margin or on the wrong side), while the $\frac{1}{2}\Vert\mathbf{w}\Vert^2$ term rewards a wide margin.

1. **Think:** If $C$ is very small (say $C = 0.01$), which term does the optimizer mostly care about? What will the margin look like, and will many or few points end up as support vectors? What about a very large $C$ (say $C = 100$)? Which one is more likely to overfit?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** In regularized logistic regression, $\lambda$ multiplied the *weight penalty*. Here, $C$ multiplies the *data* term instead. Discuss with your group: how are $C$ and $\lambda$ related?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 1.4 Coding Exercise: How C Changes the Margin

From here on, you will be writing more of the code yourself. The hints will tell you *what* to use, but not always exactly how to call it, so look back at earlier hackathons (or the scikit-learn documentation) when you need to.

**What you need to write:** Inside the loop, train a linear-kernel SVM (`SVC`) on the blob data using the loop's current value of `C`, and draw its decision boundary and margin on that loop's subplot, `ax`, with `plot_decision_boundary()`. Each subplot's title should show its value of `C`. Your code should also print how many support vectors each SVM ended up with. A fitted `SVC` keeps them in its `support_vectors_` attribute, one row per support vector.

In [ ]:
C_values = [0.01, 1, 100]
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(16, 5))

for ax, C in zip(axs, C_values):
    # --- YOUR CODE HERE ---

print('Expected: 28, 7, and 4 support vectors')
plt.show()

As $C$ grows, the margin narrows and fewer points end up inside it. With $C = 100$, the boundary has tilted to squeeze in the last few training points, so it depends on just 4 points. If one of those 4 points were an outlier or mislabeled, the whole boundary would follow it.

## 2. Kernels: When a Straight Line Won't Do

The real power of SVMs is their ability to classify data that is **not** linearly separable. Let's create a dataset with a circle inside a circle using scikit-learn's `make_circles`, and split it into training and test sets.

In [ ]:
circles_X, circles_y = datasets.make_circles(n_samples=500, noise=0.25, factor=0.5, random_state=123)

circles_X_train, circles_X_test, circles_y_train, circles_y_test = train_test_split(
    circles_X, circles_y, test_size=0.3, random_state=42, stratify=circles_y)

plt.figure(figsize=(7, 5))
plt.scatter(circles_X_train[:, 0], circles_X_train[:, 1], c=circles_y_train, cmap=plt.cm.viridis, edgecolors='k')
plt.title('Circles (Training Set)')
plt.show()

No straight line can separate these two classes. In the Logistic Regression bonus hackathon, we solved this problem by hand-crafting 28 polynomial features. SVMs use a **kernel** instead.

### 2.1 The Gaussian (RBF) Kernel

In class, we discussed **landmarks** $\mathbf{l}^{(i)}$ and a **similarity** function that measures how close a point $\mathbf{x}$ is to a landmark. The Gaussian, or **Radial Basis Function (RBF)**, kernel uses:

$$\text{similarity}(\mathbf{x}, \mathbf{l}) = \exp\Big(- \frac{\Vert\mathbf{x}-\mathbf{l}\Vert^2}{2\sigma^2}\Big) = \exp\Big(- \gamma \Vert\mathbf{x}-\mathbf{l}\Vert^2\Big), \qquad \gamma = \frac{1}{2\sigma^2}$$

The similarity is 1 when $\mathbf{x}$ sits exactly on the landmark and falls towards 0 as $\mathbf{x}$ moves away. The RBF kernel uses **every training point as a landmark**, so each point gets a new set of features: how similar it is to each training point. The SVM then draws a *linear* boundary using these new features, which becomes a *curved* boundary in the original space. (The "kernel trick" lets the SVM do this without ever building the new features explicitly.)

$\sigma^2$ controls the width of the similarity "bump". scikit-learn uses $\gamma$ (`gamma`) instead, and $\gamma$ and $\sigma^2$ are **inversely** related: a small $\sigma^2$ is a large $\gamma$.

### 2.2 Coding Exercise: Similarity Curves

**What you need to write:** First, finish `rbf_similarity()` so that it returns the RBF similarity (the $\sigma^2$ version of the formula above) between the landmark and **every** row of `X` at once, without a Python loop. `np.sum()` has an `axis` argument that lets you add up the squared differences within each row. Then, in the next cell, plot one similarity curve along `x_line` for each $\sigma^2$ in `sigma2_values`, all on the same axes and all measured from the landmark at $x = 0$. Label each curve with its $\sigma^2$ so the legend can tell them apart.

In [ ]:
def rbf_similarity(X, landmark, sigma2):
    """
    Gaussian (RBF) similarity between each row of X and a single landmark.
    X: the points, shape (m, n)
    landmark: a single point, shape (n,)
    sigma2: the width of the kernel, sigma squared
    Returns: the similarities, shape (m,)
    """
    # --- YOUR CODE HERE ---


# --- Test your function: ---
# The squared distance from (1, 2) to (0, 0) is 1 + 4 = 5, so with sigma2 = 1, the similarity is exp(-5/2) = 0.0821
print(rbf_similarity(np.array([[1.0, 2.0], [0.0, 0.0]]), np.array([0.0, 0.0]), sigma2=1.0))
print('Expected output (approx): [0.0821 1.    ]')

In [ ]:
x_line = np.linspace(-4, 4, 200).reshape(-1, 1) # 200 points along a line, shape (200, 1)
landmark = np.array([0.0])
sigma2_values = [0.25, 1, 3, 10]

plt.figure(figsize=(9, 5))

# --- YOUR CODE HERE ---

plt.xlabel('x')
plt.ylabel('similarity to the landmark at x = 0')
plt.title('RBF Similarity for Different Values of σ²')
plt.legend()
plt.show()

### Think-Pair-Share 2: The γ Hyperparameter
**Context:** $\gamma = \frac{1}{2\sigma^2}$, so a **large** $\gamma$ gives a **narrow** similarity curve, and a **small** $\gamma$ gives a **wide** one.

1. **Think:** With a very large $\gamma$, how many training points will a new point be "similar" to? Sketch what you expect the SVM's decision boundary on the circles data to look like for $\gamma = 0.01$, $\gamma = 1$, and $\gamma = 100$. Which will underfit, and which will overfit?

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Compare sketches with your group. How is an RBF SVM with a huge $\gamma$ similar to KNN with $k = 1$ from Hackathon 1?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 2.3 Coding Exercise: Logistic Regression vs. SVM Kernels

Before we tune $\gamma$, let's compare logistic regression to SVMs with linear and RBF kernels (using the default hyperparameters).

**What you need to write:** Fill in the two missing entries of `models` with a linear-kernel SVM and an RBF-kernel SVM, both with their default hyperparameters. The loop should then print each model's name next to its average accuracy across 5 cross-validation folds of the circles **training** set. `cross_val_score()` (from Hackathon 1) handles the cross-validation.

In [ ]:
models = {
    'Logistic regression': LogisticRegression(),
    'SVM, linear kernel': # --- YOUR CODE HERE ---,
    'SVM, RBF kernel': # --- YOUR CODE HERE ---,
}

print('Cross-validated mean accuracy:')
for name, model in models.items():
    # --- YOUR CODE HERE ---

print('Expected (approx): 0.46, 0.54, and 0.85')

Both linear models are stuck near 50%, which is no better than guessing. Only the RBF kernel can bend its boundary around the inner circle.

### 2.4 Coding Exercise: How γ Changes the Boundary

Time to check your Think-Pair-Share 2 sketches.

**What you need to write:** Inside the loop, train an RBF SVM on the circles **training** set using the loop's current value of `gamma`, and draw its decision boundary on that loop's subplot, `ax`. Each subplot's title should show its value of $\gamma$ along with the SVM's accuracy on both the training set and the test set, so you can see underfitting and overfitting side by side. Every scikit-learn classifier has a `.score()` method that computes its accuracy.

In [ ]:
gamma_values = [0.01, 1, 100]
fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(16, 5))

for ax, gamma in zip(axs, gamma_values):
    # --- YOUR CODE HERE ---

print('Expected (approx) train/test accuracy: 0.63/0.60, 0.85/0.85, and 0.94/0.81')
plt.show()

So, to sum it up:

- **Small $\sigma^2$ (large $\gamma$) = overfitting:** each training point only influences its immediate neighborhood, so the boundary is tailored too precisely to the training data.
- **Large $\sigma^2$ (small $\gamma$) = underfitting:** every training point influences everything, so the boundary is too general to follow the real shape of the data.

$C$ and $\gamma$ interact, so in practice we tune them **together**. That's what we'll do next, on real data.

## 3. Real Data: Handwritten Digits

The MNIST (Modified National Institute of Standards and Technology) database of handwritten digits contains 70,000 grayscale images of the digits 0 to 9, written by employees of the United States Census Bureau and by high school students. MNIST is one of the most widely used datasets in machine learning. Convolutional Neural Networks (CNNs) reach over 99% accuracy on it.

scikit-learn includes a small, low-resolution version of this dataset: 1,797 images of 8×8 pixels. Each of the 64 pixels is a feature, with a value from 0 (white) to 16 (black). Let's load it and look at a few examples.

In [ ]:
digits = datasets.load_digits()
digits_X, digits_y = digits.data, digits.target
print(f'{digits_X.shape[0]} images, each with {digits_X.shape[1]} pixel features')

fig, axs = plt.subplots(nrows=1, ncols=10, figsize=(16, 2.5))
for ax, image, label in zip(axs, digits.images, digits.target):
    ax.imshow(image, cmap='gray_r')
    ax.set(xticks=[], yticks=[], title=f'Label: {label}')
plt.show()

### Think-Pair-Share 3: Multi-Class SVMs
**Context:** An SVM finds a single boundary that separates **two** classes. The digits dataset has **ten** classes.

1. **Think:** How could you combine several *binary* (two-class) SVMs to classify the ten digits? Describe at least one strategy, and how many SVMs it would need.

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Compare strategies with your group. scikit-learn's `SVC` uses "one-vs-one". How many SVMs does it train for ten classes? What is one advantage and one disadvantage of this strategy?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 3.1 Coding Exercise: Split the Data

We will tune our hyperparameters with cross-validation on the training set, so this time we only need two sets: training and generalization.

**What you need to write:** Split `digits_X` and `digits_y` so that 75% of the images become the training set (`X_train`, `y_train`) and the other 25% become the generalization set (`X_gen`, `y_gen`). Each digit should make up the same share of both sets, and use `random_state=42` so your numbers match ours.

In [ ]:
X_train, X_gen, y_train, y_gen = # --- YOUR CODE HERE ---

print('Training set:      ', X_train.shape)
print('Generalization set:', X_gen.shape)
print('Expected shapes: (1347, 64) and (450, 64)')

### 3.2 Coding Exercise: Compare Models (Write the Whole Cell)

This time, write the whole cell yourself. Using your code from Section 2.3 as a template, compare **logistic regression**, a **linear SVM**, and an **RBF SVM** on the digits **training** set using 5-fold cross-validation.

* Use `LogisticRegression(max_iter=5000)`. The pixel values aren't scaled, so logistic regression needs more iterations than the default to converge.
* *Expected (approx):* 0.967 for logistic regression, 0.979 for the linear SVM, and 0.987 for the RBF SVM.

In [ ]:
# Compare the three models on the digits training set using 5-fold cross-validation
# --- YOUR CODE HERE ---

### 3.3 Coding Exercise: Tune the SVM with GridSearchCV

All three models do well, but let's see whether tuning $C$, $\gamma$, and the kernel can do better. We'll use `GridSearchCV` (from Hackathon 0), which tries every combination of hyperparameters using cross-validation on the training set.

A linear kernel has no $\gamma$, so instead of a single grid, we give `GridSearchCV` a **list of grids**: one for each kernel. (Otherwise, it would waste time retraining the linear SVM with every value of $\gamma$.)

**What you need to write:** Add a second grid to `param_grid` for the `'rbf'` kernel, covering every combination of `C` = 0.1, 1, 10, and 100 with `gamma` = 0.0001, 0.001, 0.01, and 0.1. Then build a `GridSearchCV` named `grid` that searches an `SVC()` over `param_grid`, judging each combination by its 5-fold cross-validated accuracy, and fit it to the training set. This takes a few seconds.

In [ ]:
param_grid = [
    {'kernel': ['linear'], 'C': [0.01, 0.1, 1, 10]},
    # --- YOUR CODE HERE ---
]

# --- YOUR CODE HERE ---

print('Best parameters:', grid.best_params_)
print(f'Best cross-validated accuracy: {grid.best_score_:.4f}')
print("Expected (approx): {'C': 10, 'gamma': 0.001, 'kernel': 'rbf'} with accuracy 0.9918")

`grid.cv_results_` stores the score of every combination. Let's look at the top 10:

In [ ]:
results = pd.DataFrame(grid.cv_results_)
results[['param_kernel', 'param_C', 'param_gamma', 'mean_test_score']].sort_values('mean_test_score', ascending=False).head(10)

**Pause point:** How many SVMs did this grid search train in total? (Hint: count the combinations in `param_grid`, the number of folds, the final refit on the whole training set, and remember Think-Pair-Share 3.)

### 3.4 Coding Exercise: Evaluate Generalization

After the search, `GridSearchCV` automatically retrains the best model on the full training set, so `grid` can make predictions directly, just like any other model.

**What you need to write:** Use the tuned model to predict the digits in the generalization set, keeping the predictions in `gen_pred` and their accuracy in `gen_acc`.

In [ ]:
# --- YOUR CODE HERE ---

print(f'Generalization accuracy: {gen_acc:.4f}')
print('Expected (approx): 0.9889')

ConfusionMatrixDisplay.from_predictions(y_gen, gen_pred)
plt.title('Tuned SVM Confusion Matrix (Generalization Set)')
plt.show()

Let's look at the handful of digits that the SVM got wrong. Would you have gotten them right?

In [ ]:
wrong = np.where(gen_pred != y_gen)[0]

fig, axs = plt.subplots(nrows=1, ncols=len(wrong), figsize=(2.5 * len(wrong), 3))
for ax, i in zip(np.atleast_1d(axs), wrong):
    ax.imshow(X_gen[i].reshape(8, 8), cmap='gray_r')
    ax.set(xticks=[], yticks=[], title=f'True: {y_gen[i]}\nPredicted: {gen_pred[i]}')
plt.show()

## 4. Decision Trees

Decision trees are a commonly used **non-parametric** supervised learning method that can be applied to both classification and regression. Instead of learning weights, a decision tree learns a series of simple yes/no questions about the features (e.g. "is the petal length less than 2.45 cm?").

A decision tree can be thought of as a tree turned upside down, with the **root** at the top. At each **node**, the tree asks a question that splits the samples into two groups that are each as *pure* (as close to a single class) as possible. The tree keeps splitting until it reaches the final layer of nodes, called the **leaves**, and each leaf predicts the most common class of the training samples that end up there.

Data scientists use decision trees because they are fast, need very little data preparation, and are **interpretable**: you can draw the tree and follow its decisions by hand.

### 4.1 Load the Iris Data

We'll use the Iris flower dataset again (from Hackathon 0). It contains 50 samples from each of 3 species of iris (0 = setosa, 1 = versicolor, 2 = virginica), described by 4 measurements: sepal length, sepal width, petal length, and petal width.

In [ ]:
iris = datasets.load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df['target'] = iris.target
display(iris_df.sample(5, random_state=42))

iris_X_train, iris_X_test, iris_y_train, iris_y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42, stratify=iris.target)

### Think-Pair-Share 4: Measuring Purity
**Context:** At each node, a decision tree chooses the question that makes the two resulting groups (the **child** nodes) as pure as possible.

1. **Think:** A node contains 40 setosa, 40 versicolor, and 40 virginica irises. Split A sends all the setosa to the left and everything else to the right. Split B sends half of each species to the left and half to the right. Which split is better, and why? Try to invent a formula that gives 0 for a perfectly pure node and larger numbers for more mixed nodes.

    *Your individual hypothesis:* **ANSWER**

2. **Pair:** Compare formulas with your group. What does your formula give for a node with two classes split 50/50? With three classes split evenly?

    *Group consensus:* **ANSWER**

3. **Share:** Be prepared to share your group's conclusion with the class.

---

### 4.2 Coding Exercise: Gini Impurity and Entropy

The two most common purity measures are the **Gini impurity** and the **entropy**, where $p_k$ is the fraction of the samples in the node that belong to class $k$:

$$\text{Gini} = 1 - \sum_k p_k^2 \qquad\qquad \text{Entropy} = -\sum_k p_k \log_2 p_k$$

The tree chooses the split with the largest **information gain**: how much the split reduces the impurity $I$ (Gini or entropy), weighting each child by its share of the samples:

$$\text{Information Gain} = I(\text{parent}) - \left(\frac{n_{\text{left}}}{n}\, I(\text{left}) + \frac{n_{\text{right}}}{n}\, I(\text{right})\right)$$

**What you need to write:** Finish `gini()` and `entropy()` so that each one takes the class labels of the samples in a node and returns that node's impurity as a single number, following the formulas above. `np.unique(y, return_counts=True)` tells you which classes appear in `y` *and* how many times each one appears, which is everything you need to find the $p_k$'s. Classes that don't appear are left out, so you don't need to worry about $\log_2 0$.

In [ ]:
def gini(y):
    """Gini impurity of a node containing the class labels y."""
    # --- YOUR CODE HERE ---


def entropy(y):
    """Entropy (in bits) of a node containing the class labels y."""
    # --- YOUR CODE HERE ---


# --- Test your functions: ---
print('Pure node:     gini =', gini([0, 0, 0, 0]), '  entropy =', entropy([0, 0, 0, 0]))
print('50/50 node:    gini =', gini([0, 0, 1, 1]), '  entropy =', entropy([0, 0, 1, 1]))
print('Iris training: gini =', round(gini(iris_y_train), 3), '  entropy =', round(entropy(iris_y_train), 3))
print('Expected: 0 and 0 (-0.0 is fine), 0.5 and 1.0, 0.667 and 1.585')

Now let's use your functions to score two candidate first splits of the iris training set (handled for you):

In [ ]:
def information_gain(y, goes_left, impurity):
    """How much a split reduces impurity. goes_left is True for each sample sent to the left child."""
    left, right = y[goes_left], y[~goes_left]
    weighted_child_impurity = (len(left) * impurity(left) + len(right) * impurity(right)) / len(y)
    return impurity(y) - weighted_child_impurity

splits = {
    'petal length <= 2.45': iris_X_train[:, 2] <= 2.45,
    'sepal width <= 3.0': iris_X_train[:, 1] <= 3.0,
}

for name, goes_left in splits.items():
    print(f'{name:22s} information gain: '
          f'{information_gain(iris_y_train, goes_left, gini):.3f} (gini), '
          f'{information_gain(iris_y_train, goes_left, entropy):.3f} (entropy)')

### 4.3 Train and Visualize the Tree

Building a whole decision tree takes just two lines with scikit-learn's `DecisionTreeClassifier`. By default, it measures purity with the Gini impurity. (Use `criterion='entropy'` to use entropy and information gain, as we discussed in class.)

We can then draw the whole tree with `plot_tree`. Each box shows the question asked at that node, the node's Gini impurity, how many training samples reached it, how many of each class (`value`), and the class it would predict.

In [ ]:
iris_tree = DecisionTreeClassifier(random_state=42)
iris_tree.fit(iris_X_train, iris_y_train)

plt.figure(figsize=(16, 10))
plot_tree(iris_tree, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.show()

print(classification_report(iris_y_test, iris_tree.predict(iris_X_test), target_names=iris.target_names))

**Pause point:** Look at the root (top) node. Which question did the tree choose to ask first? Does its Gini impurity match your `gini(iris_y_train)`, and does the choice agree with your information gain calculations above?

Notice that the tree keeps splitting until every leaf is pure (`gini = 0.0`). When left **unpruned** like this, a decision tree is likely to overfit. We often **prune** it by tuning hyperparameters such as `max_depth` (the maximum number of questions in a row). Take a look at this helpful [guide](https://towardsdatascience.com/how-to-tune-a-decision-tree-f03721801680) to the hyperparameters of a decision tree.

### 4.4 Trees vs. SVMs: Decision Boundaries

How do the boundaries of a decision tree compare to an SVM's? To draw them, we'll use only two features: sepal length and sepal width. The SVMs are wrapped in a pipeline with a `StandardScaler` (as in Hackathon 1), but the decision tree is not.

In [ ]:
iris_2d = iris.data[:, :2] # sepal length and sepal width only

models = {
    'SVM, linear kernel': make_pipeline(StandardScaler(), SVC(kernel='linear')),
    'SVM, RBF kernel': make_pipeline(StandardScaler(), SVC(kernel='rbf')),
    'Decision tree (max depth 4)': DecisionTreeClassifier(max_depth=4, random_state=42),
}

fig, axs = plt.subplots(nrows=1, ncols=3, figsize=(16, 5))
for ax, (name, model) in zip(axs, models.items()):
    cv_acc = cross_val_score(model, iris_2d, iris.target, cv=5).mean()
    model.fit(iris_2d, iris.target)
    plot_decision_boundary(model, iris_2d, iris.target, ax, title=f'{name}\ncross-validated accuracy {cv_acc:.2f}')
    ax.set(xlabel='sepal length (cm)', ylabel='sepal width (cm)')
plt.show()

**Pause point:** Why are all of the decision tree's regions rectangles? And why doesn't the decision tree need a `StandardScaler`? (Hackathon 1 asked: "Why does scaling matter more for KNN than for a decision tree?")

### 4.5 Coding Exercise: Tree Depth and Overfitting

How does a decision tree compare to the SVM on the digits data? Let's see how the tree's accuracy changes as we let it grow deeper. Following our workflow, we'll measure this with cross-validation on the **training** set, not the generalization set.

**What you need to write:** By the end of the loop, `cv_scores` should hold the 5-fold cross-validated accuracy (on the digits training set) of a decision tree limited to each `max_depth` in `depths`, and `train_scores` should hold each tree's accuracy on the training data it was fit to. Use `random_state=42` for the trees so your numbers match ours. Then plot both lists against `depths` on the same axes, labeled so the legend can tell them apart.

In [ ]:
depths = range(1, 16)
train_scores = []
cv_scores = []

for depth in depths:
    # --- YOUR CODE HERE ---

plt.figure(figsize=(9, 5))
# --- YOUR CODE HERE ---
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.title('Decision Tree Depth vs. Accuracy (Digits)')
plt.legend()
plt.show()

print(f'Best cross-validated accuracy: {max(cv_scores):.3f} at max_depth = {depths[np.argmax(cv_scores)]}')

Check your plot: the training accuracy should climb to 1.00, while the cross-validated accuracy levels off around 0.85. Compare that to the tuned SVM's 0.99!

---

## Hackathon Takeaways & Synthesis

As your group finishes this notebook, collaborate to answer the following synthesis questions.

1. **Trade-off Analysis:**
On the digits data, the tuned RBF SVM reached about 99% accuracy, while the best decision tree only reached about 85%. However, a decision tree can be printed out and followed by hand (Section 4.3), while an RBF SVM can't. Describe a real situation where you would choose the less accurate decision tree anyway. What would you be giving up?

    **ANSWER**

2. **Diagnosing Failures:**
A classmate's RBF SVM scores 100% on the training data but only 70% on new data. What is the problem? Which two hyperparameters should they change, and in which direction? What would the same problem look like for a decision tree, and how would they fix it there?

    **ANSWER**

3. **3-2-1 Summary:**
* **3** key mathematical or programming concepts your group solidified today:

    1. 

    2. 

    3. 

* **2** things you are still slightly confused about (we will address these in the next lecture!):

    1. 

    2. 

* **1** real-world scenario where you would choose an SVM, and one where you would choose a decision tree:

    1. 

### Things you *should* be able to answer:

1. How does the regularization hyperparameter ($C$) play a role in underfitting and overfitting in SVMs? How is $C$ related to the logistic regression hyperparameter $\lambda$?

2. How do the kernel hyperparameters ($C$, $\sigma^2$, $\gamma$) play a role in underfitting and overfitting in SVMs?

3. How does your choice in kernel (rbf vs linear) change based on the data?